Data Analyst: Andrei Berner Arroyo Tanta

**Project with Numpy and Pandas 1: Strategic Asset Price Simulator (GBM Model)

**Business Application:

-Need: Financial institutions require robust models to forecast potential market paths and stress-test portfolios against different economic regimes.

-Beneficiaries: Portfolio Managers, Risk Officers and Quantitative Analysts.

-Decision Support: This tool allows stakeholders to visualize long-term asset trajectories, identify volatility clusters, and allocate capital based on simulated risk-reward profiles.

**Executive Summary

This project simulates five years of market data (1,260$ trading days) for three distinct asset classes using the Geometric Brownian Motion (GBM) model. It demonstrates a seamless transition from a high-performance NumPy computational engine to a structured Pandas analytical layer (for the result, scroll down until the last code).

In [18]:
#Model Configuration & Parameters
import numpy as np
import pandas as pd

In [19]:
#Reproducibility and time parameters
np.random.seed(42)
N_YEARS = 5
N_DAYS_YEAR = 252
N_STEPS = N_DAYS_YEAR * N_YEARS
DT = 1 / N_DAYS_YEAR

#Asset configurations (Tickers and expected behavior)
asset_config = {
    "GROWTH_STK": {'initial': 150.25, 'mu': 0.2575, 'sigma': 0.22}, #it will tend to increase 
    "FLAT_STK":   {'initial': 188.85, 'mu': 0.00015, 'sigma': 0.045}, #it will tend to be the same
    "BEAR_STK":   {'initial': 271.01, 'mu': -0.658,  'sigma': 0.87} #it will tend to decrease 
}

tickers = list(asset_config.keys())
initial_prices = np.array([v["initial"] for v in asset_config.values()])
mus = np.array([v["mu"] for v in asset_config.values()])
sigmas = np.array([v["sigma"] for v in asset_config.values()])

In [20]:
#Performance Layer: NumPy Engine

#Generating Gaussian Noise (Market Shocks)
shocks = np.random.standard_normal(size=(N_STEPS, len(tickers))).astype(np.float32)

#Brownian Motion Process
#We calculate the cumulative sum of shocks scaled by the square root of DT
brownian_motion = np.cumsum(shocks, axis=0) * np.sqrt(DT)

#Applying the GBM formula
time_vector = (np.arange(1, N_STEPS + 1) * DT).reshape(-1, 1)
drift = (mus - 0.5 * sigmas**2) * time_vector
diffusion = sigmas * brownian_motion

prices_matrix = initial_prices * np.exp(drift + diffusion)

print(f"Simulation Complete: {prices_matrix.shape} matrix generated using NumPy.")

Simulation Complete: (1260, 3) matrix generated using NumPy.


In [21]:
#Semantic Layer: Pandas Structuring

#Create a Business Day date range
date_index = pd.date_range(start="2020-01-01", periods=N_STEPS, freq="B")

#Build the primary DataFrame
df_stocks = pd.DataFrame(
    data=prices_matrix,
    index=date_index,
    columns=tickers
).astype("float32")

#Inject metadata for traceability
df_stocks.attrs['project_name'] = "Financial Asset Simulator v1.0"
df_stocks.index.name = "Date"
df_stocks.columns.name = "Ticker"

print("Pandas data structure successfully created.")
df_stocks.head()

Pandas data structure successfully created.


Ticker,GROWTH_STK,FLAT_STK,BEAR_STK
Date,,,
2020-01-01,151.427994,188.775345,279.650146
2020-01-02,154.801422,188.649445,274.951447
2020-01-03,158.373291,189.059647,266.867462
2020-01-06,159.716400,188.810806,259.074341
2020-01-07,160.401306,187.788895,234.737595


In [22]:
#Risk Analysis & Insights

#Calculate absolute daily percentage returns
daily_changes = np.abs(np.diff(prices_matrix, axis=0) / prices_matrix[:-1]) * 100

#Identify events where change exceeded 10%
risk_mask = daily_changes > 10
total_alerts = np.sum(risk_mask)
risk_pct = (total_alerts / prices_matrix.size) * 100

print(f"--- Report ---")
print(f"Volatility Alerts (>10%): {total_alerts} events detected.")
print(f"Dataset Risk Index: {np.round(risk_pct,3)}%")

--- Report ---
Volatility Alerts (>10%): 85 events detected.
Dataset Risk Index: 2.249%


Volatility Alerts: The system detected 85 high-volatility events where daily prices shifted by more than 10%.
Risk Index: The total dataset carries a 2.249% risk index based on extreme price movements.

In [23]:
#Correlation Analysis
#Pearson Correlation Matrix
correlation_matrix = df_stocks.corr()

print(f"--- Report ---")
print(f"\nInter-Asset Correlation Matrix:")
display(correlation_matrix.style.background_gradient(cmap='coolwarm').format(precision=3))

--- Report ---

Inter-Asset Correlation Matrix:


Ticker,GROWTH_STK,FLAT_STK,BEAR_STK
Ticker,,,
GROWTH_STK,1.000,0.318,-0.868
FLAT_STK,0.318,1.000,-0.345
BEAR_STK,-0.868,-0.345,1.000


A strong negative correlation (-0.868) was identified between Growth and Bear assets, suggesting a natural hedge for portfolio diversification.

In [24]:
#Executive Dashboard
#Compile summary metrics
summary_metrics = pd.DataFrame({
    "Start Price": df_stocks.iloc[0],
    "End Price": df_stocks.iloc[-1],
    "Min Price": df_stocks.min(),
    "Max Price": df_stocks.max(),
    "Total Return (%)": ((df_stocks.iloc[-1] - df_stocks.iloc[0]) / df_stocks.iloc[0]) * 100
}).T

print(f"--- Report ---")
summary_metrics

--- Report ---


Ticker,GROWTH_STK,FLAT_STK,BEAR_STK
Start Price,151.427994,188.775345,279.650146
End Price,763.306702,184.747650,58.437901
Min Price,146.292191,169.147446,29.622076
Max Price,873.420959,191.048203,397.601593
Total Return (%),404.072388,-2.133591,-79.103210
